In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master("local[*]").appName("CustomerSegmentation").getOrCreate()

In [4]:
dataset = spark.read.csv(
    'customer_segmentation_dataset.csv',
    inferSchema=True,
    header=True
)

In [5]:
dataset.printSchema()

root
 |-- age: integer (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- spending_score: integer (nullable = true)
 |-- years_as_customer: integer (nullable = true)
 |-- number_of_purchases: integer (nullable = true)
 |-- customer_segment: string (nullable = true)



In [6]:
dataset.select('customer_segment').distinct().show(10)
dataset.count()

+----------------+
|customer_segment|
+----------------+
|    Medium Value|
|      High Value|
|       Low Value|
+----------------+



219716

In [7]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler

In [8]:
feature_cols = ["age", "annual_income", "spending_score",
                 "years_as_customer", "number_of_purchases"]

In [9]:
vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [10]:
df_temp = vector_assembler.transform(dataset)
df_temp.show(3)

+---+-------------+--------------+-----------------+-------------------+----------------+--------------------+
|age|annual_income|spending_score|years_as_customer|number_of_purchases|customer_segment|            features|
+---+-------------+--------------+-----------------+-------------------+----------------+--------------------+
| 22|       113441|            19|                2|                 44|       Low Value|[22.0,113441.0,19...|
| 47|        85415|            74|                7|                 11|    Medium Value|[47.0,85415.0,74....|
| 60|        78075|            18|               19|                 37|       Low Value|[60.0,78075.0,18....|
+---+-------------+--------------+-----------------+-------------------+----------------+--------------------+
only showing top 3 rows


In [11]:
df = df_temp.select("customer_segment", "features")
df.show(3)

+----------------+--------------------+
|customer_segment|            features|
+----------------+--------------------+
|       Low Value|[22.0,113441.0,19...|
|    Medium Value|[47.0,85415.0,74....|
|       Low Value|[60.0,78075.0,18....|
+----------------+--------------------+
only showing top 3 rows


In [12]:
from pyspark.ml.feature import StringIndexer

l_indexer = StringIndexer(inputCol="customer_segment", outputCol="labelIndex")
df = l_indexer.fit(df).transform(df)

In [13]:
df.select('customer_segment', 'labelIndex').distinct().show(3)

+----------------+----------+
|customer_segment|labelIndex|
+----------------+----------+
|      High Value|       2.0|
|    Medium Value|       1.0|
|       Low Value|       0.0|
+----------------+----------+



In [14]:
(trainingData, testData) = df.randomSplit([0.7, 0.3], seed=1234)

In [15]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [16]:
# Build a single Decision Tree classifier. This is our baseline model,
# used later for comparison against the bagged (Random Forest) version.
dt = DecisionTreeClassifier(
    labelCol="labelIndex",
    featuresCol="features",
    impurity='entropy',
    maxDepth=5,
    seed=1234
)


In [17]:
# Fit the tree on the training data.
dt_model = dt.fit(trainingData)

In [18]:
# Generate predictions on the unseen test data.
dt_predictions = dt_model.transform(testData)

In [19]:
# MulticlassClassificationEvaluator computes performance metrics
# (accuracy here) for problems with more than two classes, by comparing
# the "prediction" column against the true "labelIndex" column.
evaluator = MulticlassClassificationEvaluator(
    labelCol="labelIndex",
    predictionCol="prediction",
    metricName="accuracy"
)

dt_accuracy = evaluator.evaluate(dt_predictions)
print("Decision Tree Test Accuracy =", dt_accuracy)
print(dt_model.toDebugString)

Decision Tree Test Accuracy = 0.9796644172898048
DecisionTreeClassificationModel: uid=DecisionTreeClassifier_d4605e6962fc, depth=2, numNodes=5, numClasses=3, numFeatures=5
  If (feature 2 <= 49.5)
   Predict: 0.0
  Else (feature 2 > 49.5)
   If (feature 2 <= 76.5)
    Predict: 1.0
   Else (feature 2 > 76.5)
    Predict: 2.0



In [20]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="labelIndex",
    featuresCol="features",
    numTrees=25,              # number of bootstrap-sampled trees (bagging size)
    maxDepth=5,
    featureSubsetStrategy="sqrt",  # extra randomisation on top of bagging
    seed=1234
)

In [21]:
# Train the bagged ensemble of decision trees.
rf_model = rf.fit(trainingData)

In [24]:
# Train the bagged ensemble of decision trees.
rf_model = rf.fit(trainingData)

In [25]:
# Generate predictions on the test set using the ensemble's majority vote.
rf_predictions = rf_model.transform(testData)

In [26]:

rf_accuracy = evaluator.evaluate(rf_predictions)
print("Random Forest (Bagging) Test Accuracy =", rf_accuracy)

Random Forest (Bagging) Test Accuracy = 0.9796644172898048


In [27]:
# all trees in the bagged ensemble.
print("Feature Importances:", list(zip(feature_cols, rf_model.featureImportances.toArray())))

Feature Importances: [('age', np.float64(6.1669507966043e-05)), ('annual_income', np.float64(4.755154040219125e-05)), ('spending_score', np.float64(0.9998249423065287)), ('years_as_customer', np.float64(1.657543337498528e-05)), ('number_of_purchases', np.float64(4.926121172802391e-05))]


In [29]:
# MODEL 3: MULTICLASS LOGISTIC REGRESSION (One-vs-Rest, no bagging)
from pyspark.ml.classification import OneVsRest, LogisticRegression

# Re-split with a different seed, matching the style of the Iris example.
train, test = df.randomSplit([0.7, 0.3], seed=2018)

lr = LogisticRegression(
    maxIter=100,
    featuresCol="features",
    labelCol='labelIndex'
)

In [30]:
# OneVsRest turns the binary Logistic Regression algorithm into a
# multiclass classifier by training one binary model per class
# ("this class" vs "all other classes").
ovr = OneVsRest(classifier=lr, labelCol='labelIndex', featuresCol='features')

ovr_model = ovr.fit(train)
ovr_predictions = ovr_model.transform(test)

ovr_accuracy = evaluator.evaluate(ovr_predictions)
print("Logistic Regression (OneVsRest) Test Accuracy =", ovr_accuracy)


Logistic Regression (OneVsRest) Test Accuracy = 1.0


In [31]:
import pyspark.sql.functions as F
from pyspark.sql import Window

n_estimators = 5       # number of bootstrap models in the bagging ensemble
bagged_models = []

for i in range(n_estimators):
    bootstrap_sample = train.sample(withReplacement=True, fraction=1.0, seed=i)
    lr_i = LogisticRegression(maxIter=100, featuresCol="features", labelCol='labelIndex')
    ovr_i = OneVsRest(classifier=lr_i, labelCol='labelIndex', featuresCol='features')
    model_i = ovr_i.fit(bootstrap_sample)

    bagged_models.append(model_i)
    print(f"Trained bagging estimator {i + 1}/{n_estimators}")

Trained bagging estimator 1/5
Trained bagging estimator 2/5
Trained bagging estimator 3/5
Trained bagging estimator 4/5
Trained bagging estimator 5/5


In [32]:
test_indexed = test.withColumn("row_id", F.monotonically_increasing_id()).cache()

In [33]:
predictions_combined = test_indexed.select("row_id", "labelIndex")

In [34]:
for idx, model_i in enumerate(bagged_models):
    pred_i = model_i.transform(test_indexed).select(
        "row_id",
        F.col("prediction").alias(f"pred_{idx}")
    )
    predictions_combined = predictions_combined.join(pred_i, on="row_id", how="inner")

In [35]:
pred_columns = [f"pred_{idx}" for idx in range(n_estimators)]
votes_long = predictions_combined.select(
    "row_id",
    "labelIndex",
    F.explode(F.array(*pred_columns)).alias("vote")
)

In [36]:
# Count how many models voted for each class, per test row.
vote_counts = votes_long.groupBy("row_id", "labelIndex", "vote").count()

In [37]:
# For each row, rank the candidate classes by vote count (highest first)
# and keep only the top-ranked class as the majority-vote prediction.
majority_window = Window.partitionBy("row_id").orderBy(F.desc("count"))
bagged_predictions = (
    vote_counts
    .withColumn("rank", F.row_number().over(majority_window))
    .filter(F.col("rank") == 1)
    .select("row_id", "labelIndex", F.col("vote").alias("bagged_prediction"))
)

In [38]:
# Evaluate the manually bagged Logistic Regression ensemble's accuracy.
bagged_lr_evaluator = MulticlassClassificationEvaluator(
    labelCol="labelIndex",
    predictionCol="bagged_prediction",
    metricName="accuracy"
)
bagged_lr_accuracy = bagged_lr_evaluator.evaluate(bagged_predictions)
print("Manually Bagged Logistic Regression Test Accuracy =", bagged_lr_accuracy)


Manually Bagged Logistic Regression Test Accuracy = 1.0


In [39]:
print("Model Accuracy Comparison")
print("--------------------------------------------------")
print(f"Decision Tree (single, no bagging):       {dt_accuracy:.4f}")
print(f"Random Forest (built-in bagging):         {rf_accuracy:.4f}")
print(f"Logistic Regression OneVsRest (no bagging): {ovr_accuracy:.4f}")
print(f"Logistic Regression OneVsRest (manual bagging): {bagged_lr_accuracy:.4f}")

Model Accuracy Comparison
--------------------------------------------------
Decision Tree (single, no bagging):       0.9797
Random Forest (built-in bagging):         0.9797
Logistic Regression OneVsRest (no bagging): 1.0000
Logistic Regression OneVsRest (manual bagging): 1.0000
